# M02 — Run and Interrogate Your First ML System

Run one complete supervised ML system before studying its mechanisms. This notebook is CPU-only, deterministic where practical, and uses a committed local dataset: no network, paid API, or secret is required.

## The whole system

raw CSV → features/target → train/test split → model fit → predictions → evaluation → interrogation

Treat this as a map of information flow. Test labels stay outside fitting. They re-enter only when predictions are evaluated.

## Prediction-before-action protocol

Before each experiment, write:
- the direction you expect the evidence to move;
- why;
- the one dimension being changed;
- what is held constant.

Run the next cell only after recording that prediction outside this source notebook. Then compare prediction with evidence; do not rewrite the prediction after seeing the result.

In [ ]:
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (
    ConfusionMatrixDisplay,
    accuracy_score,
    balanced_accuracy_score,
    classification_report,
    confusion_matrix,
)
from sklearn.model_selection import StratifiedKFold, cross_val_score, train_test_split
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.tree import DecisionTreeClassifier

RANDOM_STATE = 7
pd.set_option("display.max_columns", 20)
pd.set_option("display.precision", 3)

def find_repository_root(start):
    start = Path(start).resolve()
    for candidate in (start, *start.parents):
        if (candidate / "missions" / "M02" / "manifest.yaml").is_file():
            return candidate
    raise FileNotFoundError("Run this notebook from inside the LearningOS-AI repository.")

ROOT = find_repository_root(Path.cwd())
DATA_PATH = ROOT / "datasets" / "M02" / "wine.csv"
print({"repository_root": str(ROOT), "data_path": str(DATA_PATH), "seed": RANDOM_STATE})

## 1. Raw data

The raw-data boundary is a committed CSV. Inspect shape, schema, missingness, target counts, and sample rows before deciding what the model may consume.

In [ ]:
raw_data = pd.read_csv(DATA_PATH)

EXPECTED_COLUMNS = [
    "alcohol",
    "malic_acid",
    "ash",
    "alcalinity_of_ash",
    "magnesium",
    "total_phenols",
    "flavanoids",
    "nonflavanoid_phenols",
    "proanthocyanins",
    "color_intensity",
    "hue",
    "od280_od315_of_diluted_wines",
    "proline",
    "target",
]
assert raw_data.shape == (178, 14)
assert raw_data.columns.tolist() == EXPECTED_COLUMNS
assert raw_data.isna().sum().sum() == 0
assert set(raw_data["target"]) == {0, 1, 2}
raw_data.head()

In [ ]:
raw_summary = pd.DataFrame(
    {
        "dtype": raw_data.dtypes.astype(str),
        "missing": raw_data.isna().sum(),
        "unique": raw_data.nunique(),
    }
)
print("Class counts")
print(raw_data["target"].value_counts().sort_index())
raw_summary

## 2. Features and target

The 13 chemical measurements are features. The target is the wine class. Keeping the target out of the feature list prevents a direct label leak.

In [ ]:
FEATURE_COLUMNS = [column for column in raw_data.columns if column != "target"]
TARGET_COLUMN = "target"

X = raw_data[FEATURE_COLUMNS].copy()
y = raw_data[TARGET_COLUMN].astype(int).copy()

assert X.shape == (178, 13)
assert TARGET_COLUMN not in X.columns
assert np.isfinite(X.to_numpy()).all()
print({"feature_shape": X.shape, "target_shape": y.shape, "classes": sorted(y.unique())})

### Predict before running — baseline

Predict whether held-out accuracy will be above, near, or below 0.80. State why scaling plus logistic regression might succeed or fail, and name the evidence that could prove your prediction wrong.

In [ ]:
all_indices = raw_data.index.to_numpy()
(
    X_train,
    X_test,
    y_train,
    y_test,
    train_indices,
    test_indices,
) = train_test_split(
    X,
    y,
    all_indices,
    test_size=0.25,
    random_state=RANDOM_STATE,
    stratify=y,
)

assert set(train_indices).isdisjoint(set(test_indices))
assert len(train_indices) + len(test_indices) == len(raw_data)
split_balance = pd.DataFrame(
    {
        "all": y.value_counts(normalize=True).sort_index(),
        "train": y_train.value_counts(normalize=True).sort_index(),
        "test": y_test.value_counts(normalize=True).sort_index(),
    }
)
print({"train_rows": len(X_train), "test_rows": len(X_test)})
split_balance

## 3. Model recipe

A pipeline binds scaling to classification. Fitting the pipeline on training rows means scaling statistics are also learned from training rows only.

In [ ]:
baseline_model = make_pipeline(
    StandardScaler(),
    LogisticRegression(C=1.0, max_iter=2000, random_state=RANDOM_STATE),
)
baseline_model

## 4. Fitting boundary

The next call is the learning boundary. It may read training features and training labels. It must not read test labels or preprocessing statistics computed from test rows.

In [ ]:
fit_boundary = "baseline_model.fit(X_train, y_train)"
baseline_model.fit(X_train, y_train)
print("Fitted steps:", list(baseline_model.named_steps))

## 5. Prediction boundary

Prediction consumes held-out features and fitted state. Notice that the call does not need y_test.

In [ ]:
prediction_boundary = "baseline_model.predict(X_test)"
baseline_predictions = baseline_model.predict(X_test)

assert len(baseline_predictions) == len(X_test)
assert set(baseline_predictions).issubset(set(y.unique()))
baseline_predictions[:10]

## 6. Honest evaluation boundary

Ground truth re-enters only now. Accuracy gives the fraction correct; balanced accuracy averages class recall; the confusion matrix retains which classes were confused.

In [ ]:
evaluation_boundary = "accuracy_score(y_test, baseline_predictions)"
baseline_accuracy = accuracy_score(y_test, baseline_predictions)
baseline_balanced_accuracy = balanced_accuracy_score(y_test, baseline_predictions)
baseline_confusion = confusion_matrix(y_test, baseline_predictions)

baseline_metrics = pd.Series(
    {
        "accuracy": baseline_accuracy,
        "balanced_accuracy": baseline_balanced_accuracy,
        "test_rows": len(y_test),
    },
    name="baseline",
)
print(baseline_metrics)
print("\nClassification report")
print(pd.DataFrame(classification_report(y_test, baseline_predictions, output_dict=True)).T)
baseline_confusion

In [ ]:
ConfusionMatrixDisplay.from_predictions(
    y_test,
    baseline_predictions,
    display_labels=["class 0", "class 1", "class 2"],
    cmap="Blues",
    colorbar=False,
)
plt.title("Baseline held-out confusion matrix")
plt.tight_layout()
plt.show()

## 7. Interrogation

A score compresses behavior. Recover row-level errors and inspect which standardized coefficients are largest in this fitted linear model. Coefficient magnitude is a model diagnostic, not causal importance.

In [ ]:
error_rows = raw_data.loc[test_indices].copy()
error_rows["actual"] = y_test.to_numpy()
error_rows["predicted"] = baseline_predictions
error_rows["correct"] = error_rows["actual"] == error_rows["predicted"]
errors_only = error_rows.loc[
    ~error_rows["correct"],
    ["actual", "predicted", *FEATURE_COLUMNS],
]
print({"errors": len(errors_only), "correct": int(error_rows["correct"].sum())})
errors_only.head(10)

In [ ]:
classifier = baseline_model.named_steps["logisticregression"]
coefficient_strength = pd.Series(
    np.abs(classifier.coef_).mean(axis=0),
    index=FEATURE_COLUMNS,
    name="mean_absolute_standardized_coefficient",
).sort_values(ascending=False)
coefficient_strength.head(8)

## Controlled experiments

The following helper repeats the same honest holdout protocol. Each experiment section names the dimension that changes. Predictions must be recorded before its code cell runs.

In [ ]:
def make_logistic(C=1.0):
    return make_pipeline(
        StandardScaler(),
        LogisticRegression(C=C, max_iter=2000, random_state=RANDOM_STATE),
    )

def score_holdout(model, X_train_part, X_test_part, y_train_part, y_test_part):
    model.fit(X_train_part, y_train_part)
    train_predictions = model.predict(X_train_part)
    test_predictions = model.predict(X_test_part)
    return {
        "train_accuracy": accuracy_score(y_train_part, train_predictions),
        "test_accuracy": accuracy_score(y_test_part, test_predictions),
        "balanced_test_accuracy": balanced_accuracy_score(y_test_part, test_predictions),
    }

### Predict before running — Experiment 1: split

Predict how much the score will move when only the sampled rows or test proportion changes. Constants: dataset, all 13 features, logistic model, scaling, C=1.0, and accuracy.

In [ ]:
split_records = []
for scenario, split_seed, test_fraction in [
    ("baseline", 7, 0.25),
    ("different_rows", 42, 0.25),
    ("larger_test", 7, 0.30),
]:
    X_train_part, X_test_part, y_train_part, y_test_part = train_test_split(
        X,
        y,
        test_size=test_fraction,
        random_state=split_seed,
        stratify=y,
    )
    record = score_holdout(
        make_logistic(),
        X_train_part,
        X_test_part,
        y_train_part,
        y_test_part,
    )
    record.update(
        {
            "scenario": scenario,
            "split_seed": split_seed,
            "test_fraction": test_fraction,
            "test_rows": len(y_test_part),
        }
    )
    split_records.append(record)

split_results = pd.DataFrame(split_records).set_index("scenario")
split_results

### Predict before running — Experiment 2: features

Predict whether using only flavanoids and proline will preserve the full-feature score. Constants: baseline rows, scaling, logistic model, C=1.0, and evaluation.

In [ ]:
TWO_FEATURES = ["flavanoids", "proline"]
feature_records = []

for feature_set, columns in [
    ("all_13", FEATURE_COLUMNS),
    ("two_features", TWO_FEATURES),
]:
    result = score_holdout(
        make_logistic(),
        X_train[columns],
        X_test[columns],
        y_train,
        y_test,
    )
    result.update({"feature_set": feature_set, "feature_count": len(columns)})
    feature_records.append(result)

feature_results = pd.DataFrame(feature_records).set_index("feature_set")
feature_results

### Predict before running — Experiment 3: model

Predict whether a depth-3 decision tree will beat logistic regression on the same rows and features. Constants: data, split, scaling step, and evaluation.

In [ ]:
model_candidates = {
    "logistic_regression": make_logistic(),
    "depth_3_tree": make_pipeline(
        StandardScaler(),
        DecisionTreeClassifier(max_depth=3, random_state=RANDOM_STATE),
    ),
}
model_records = []
for model_name, candidate in model_candidates.items():
    result = score_holdout(candidate, X_train, X_test, y_train, y_test)
    result["model"] = model_name
    model_records.append(result)

model_results = pd.DataFrame(model_records).set_index("model")
model_results["train_test_gap"] = (
    model_results["train_accuracy"] - model_results["test_accuracy"]
)
model_results

### Predict before running — Experiment 4: selected hyperparameter

C controls logistic-regression regularization strength: smaller C means stronger regularization. Predict the train and test pattern. Constants: dataset, features, split, scaling, model family, and evaluation.

In [ ]:
hyperparameter_records = []
for C_value in [0.01, 0.1, 1.0, 100.0]:
    result = score_holdout(
        make_logistic(C=C_value),
        X_train,
        X_test,
        y_train,
        y_test,
    )
    result["C"] = C_value
    hyperparameter_records.append(result)

hyperparameter_results = pd.DataFrame(hyperparameter_records).set_index("C")
hyperparameter_results

### Predict before running — Experiment 5 and controlled failure A: labels

Predict the honest held-out score after training labels are deterministically shuffled. Constants: feature rows, split, scaling, model, C, and test truth. What diagnostic distinguishes corrupted supervision from a coding crash?

In [ ]:
label_rng = np.random.default_rng(42)
corrupted_y_train = pd.Series(
    label_rng.permutation(y_train.to_numpy()),
    index=y_train.index,
    name="corrupted_target",
)
label_disagreement_rate = float(
    np.mean(corrupted_y_train.to_numpy() != y_train.to_numpy())
)

corrupted_label_model = make_logistic()
corrupted_label_model.fit(X_train, corrupted_y_train)
corrupted_label_predictions = corrupted_label_model.predict(X_test)
corrupted_label_accuracy = accuracy_score(y_test, corrupted_label_predictions)

label_results = pd.Series(
    {
        "baseline_accuracy": baseline_accuracy,
        "corrupted_label_accuracy": corrupted_label_accuracy,
        "label_disagreement_rate": label_disagreement_rate,
    }
)
assert corrupted_label_accuracy < baseline_accuracy
label_results

### Predict before running — Experiment 6: evaluation setup

Predict how five stratified fold scores will compare with the single baseline holdout. Constants: dataset, all features, preprocessing, model, C, and accuracy. The changed dimension is how evaluation rows are rotated.

In [ ]:
cross_validation = StratifiedKFold(
    n_splits=5,
    shuffle=True,
    random_state=RANDOM_STATE,
)
cv_scores = cross_val_score(
    make_logistic(),
    X,
    y,
    cv=cross_validation,
    scoring="accuracy",
    n_jobs=1,
)
evaluation_results = pd.Series(
    {
        "holdout_accuracy": baseline_accuracy,
        "cv_mean_accuracy": cv_scores.mean(),
        "cv_standard_deviation": cv_scores.std(ddof=1),
        "cv_minimum": cv_scores.min(),
        "cv_maximum": cv_scores.max(),
    }
)
print("Fold scores:", np.round(cv_scores, 3))
evaluation_results

### Predict before running — Controlled failure B: invalid evaluation

Predict the result of comparing baseline_predictions with itself. Then explain why the returned number is not evidence about correctness even though the metric function executes without error.

In [ ]:
invalid_accuracy = accuracy_score(
    baseline_predictions,
    baseline_predictions,
)
repaired_accuracy = accuracy_score(y_test, baseline_predictions)

assert invalid_accuracy == 1.0
assert repaired_accuracy == baseline_accuracy
pd.Series(
    {
        "invalid_self_comparison": invalid_accuracy,
        "repaired_held_out_evaluation": repaired_accuracy,
    }
)

## Code reading: name the boundaries

Locate each exact call below. For each one, state its inputs, output or learned state, and which information is forbidden. Fitting changes model state; prediction consumes fitted state without labels; evaluation compares aligned truth and predictions without changing the model.

In [ ]:
boundary_map = pd.DataFrame(
    [
        {
            "boundary": "fit",
            "call": fit_boundary,
            "allowed_information": "training features and training labels",
            "forbidden_information": "test labels and test-derived preprocessing",
        },
        {
            "boundary": "prediction",
            "call": prediction_boundary,
            "allowed_information": "fitted state and held-out features",
            "forbidden_information": "held-out truth",
        },
        {
            "boundary": "evaluation",
            "call": evaluation_boundary,
            "allowed_information": "aligned held-out truth and predictions",
            "forbidden_information": "self-comparison presented as correctness",
        },
    ]
).set_index("boundary")
boundary_map

## Executable validation

These assertions are mission invariants, not proof of production quality. They check data integrity, split separation, prediction shape, honest metric range, the expected controlled-failure direction, and evaluation repair.

In [ ]:
assert raw_data.shape == (178, 14)
assert X.shape[1] == 13
assert set(train_indices).isdisjoint(set(test_indices))
assert len(baseline_predictions) == len(y_test)
assert 0.0 <= baseline_accuracy <= 1.0
assert baseline_accuracy >= 0.80
assert 0.0 <= baseline_balanced_accuracy <= 1.0
assert corrupted_label_accuracy < baseline_accuracy
assert invalid_accuracy == 1.0
assert repaired_accuracy == baseline_accuracy
assert len(cv_scores) == 5
print("M02 executable invariants passed.")

In [ ]:
experiment_summary = pd.DataFrame(
    [
        {
            "dimension": "split",
            "baseline": baseline_accuracy,
            "comparison": split_results.loc["different_rows", "test_accuracy"],
        },
        {
            "dimension": "features",
            "baseline": feature_results.loc["all_13", "test_accuracy"],
            "comparison": feature_results.loc["two_features", "test_accuracy"],
        },
        {
            "dimension": "model",
            "baseline": model_results.loc["logistic_regression", "test_accuracy"],
            "comparison": model_results.loc["depth_3_tree", "test_accuracy"],
        },
        {
            "dimension": "hyperparameter",
            "baseline": hyperparameter_results.loc[1.0, "test_accuracy"],
            "comparison": hyperparameter_results.loc[0.01, "test_accuracy"],
        },
        {
            "dimension": "labels",
            "baseline": baseline_accuracy,
            "comparison": corrupted_label_accuracy,
        },
        {
            "dimension": "evaluation_setup",
            "baseline": baseline_accuracy,
            "comparison": cv_scores.mean(),
        },
    ]
).set_index("dimension")
experiment_summary["delta"] = (
    experiment_summary["comparison"] - experiment_summary["baseline"]
)
experiment_summary

## Diagnose, do not merely describe

For each controlled failure, record:
1. hypothesis;
2. observed evidence;
3. root cause at the data or evaluation boundary;
4. smallest repair;
5. verification;
6. one recurrence check.

A low or perfect score is an observation. A diagnosis explains why the evidence became untrustworthy and how the repair restores the intended information relationship.

## Limitations

This small, clean benchmark is suitable for orientation, not production claims. One holdout is sensitive to sampled rows; cross-validation still estimates only performance under this dataset and protocol. Coefficients depend on scaling and the fitted linear model and are not causal effects. Synthetic shuffling is easier to detect than sparse or systematic real label error. Accuracy can hide asymmetric consequences that this mission does not model.

## No-AI gate — fresh supervised run

Stop before treating this notebook as completion evidence. In a blank notebook or script, without AI-generated code or copied cells, build a fresh supervised run from the local CSV. Change at least two of split seed, feature set, model, or selected hyperparameter. Record a prediction first; validate disjoint rows and prediction count; report two metrics and an error interrogation; identify fit, prediction, and evaluation boundaries; reproduce one controlled failure and repair it; explain what the evidence does and does not establish.

## V00 integration

You have now operated the first V00 vertical slice: data became learned state, predictions, and bounded evidence. The next standard is independent transfer. A successful review depends on reproducible execution, experimental records, failure diagnosis, limitations, and the fresh no-AI artifact—not on a high score alone.